\*Please Run in colab

# Setup

### Environment Setup

In [ ]:
!pip install instructor

In [ ]:
# !cp /content/macro_financial_forecasting/applications/macro_financial_forecasting/data/danidanou_Bloomberg_Financial_News_train /content/
!cd /content
!rm -rf macro_financial_forecasting

In [1]:
!git clone https://github.com/chuanbinp/macro_financial_forecasting.git

Cloning into 'macro_financial_forecasting'...
remote: Enumerating objects: 1937, done.
remote: Counting objects: 100% (647/647), done.
remote: Compressing objects: 100% (208/208), done.
remote: Total 1937 (delta 496), reused 441 (delta 439), pack-reused 1290 (from 2)
Receiving objects: 100% (1937/1937), 39.58 MiB | 12.64 MiB/s, done.
Resolving deltas: 100% (1238/1238), done.


In [2]:
# !cp /content/danidanou_Bloomberg_Financial_News_train /content/macro_financial_forecasting/applications/macro_financial_forecasting/data/danidanou_Bloomberg_Financial_News_train

In [3]:
%cd macro_financial_forecasting/applications/macro_financial_forecasting/src

/content/macro_financial_forecasting/applications/macro_financial_forecasting/src


### Mount GDrive

In [ ]:
from google.colab import drive

# This will prompt you to authorize Colab to access your Google Drive.
drive.mount('/content/gdrive')
GDRIVE_PATH = "/content/gdrive/MyDrive/macro_financial_forecasting_files/"

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


### Code Setup

In [4]:
from config import Config
from train_data_loader import TrainDataLoader
from data_model.bloomberg_news_entry import BloombergNewsEntry
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
config = Config("../config.env")

train_data_loader = TrainDataLoader(config)

# Train Data Loader

In [ ]:
print("Starting loading pipeline ...")
print(f"Config: {config}")

train_ds = train_data_loader.load()
print("Loading pipeline completed.")

Starting loading pipeline ...
Config: Config(
  gemini_api_key: !secret!
  openai_api_key: !secret!
  llm_model: openai/gpt-5-nano-2025-08-07
  industries: ['Information Technology', 'Health Care', 'Financials', 'Consumer Discretionary', 'Communication Services', 'Industrials', 'Consumer Staples', 'Energy', 'Utilities', 'Real Estate', 'Materials', 'General Market', 'None']
  dataset_name: danidanou/Bloomberg_Financial_News
  dataset_dir: ../data/
  rss_feeds: ['https://feeds.bloomberg.com/news/news.rss', 'https://feeds.bloomberg.com/markets/news.rss', 'https://feeds.bloomberg.com/business/news.rss', 'https://feeds.bloomberg.com/technology/news.rss', 'https://feeds.bloomberg.com/politics/news.rss', 'https://feeds.bloomberg.com/wealth/news.rss', 'https://feeds.bloomberg.com/economics/news.rss', 'https://feeds.bloomberg.com/green/news.rss', 'https://feeds.bloomberg.com/pursuits/news.rss', 'https://feeds.bloomberg.com/opinion/news.rss', 'https://feeds.bloomberg.com/finance/news.rss', 'http

# Data Processing Pipeline

Update these 2 variable to specify which indices to process.

In [ ]:
DATA_START=0
DATA_END=50000

In [ ]:
from processor import NewsProcessor
import nest_asyncio
nest_asyncio.apply()

processor = NewsProcessor(config)
sample = processor.remove_redundant_info(train_ds[DATA_START:DATA_END])
df = processor.enrich_news_entries_with_classifications(sample, save_path=f"{GDRIVE_PATH}processed_news") #Sample size
df = processor.group_by_date_and_industry(df, save_path=f"{GDRIVE_PATH}grouped_news")
df = processor.filter_and_analyze_news(df)
df = processor.extract_impactful_news(df, top_n=3, save_path=f"{GDRIVE_PATH}impact_news")
df = processor.get_consolidated_sentiment(df, save_path=f"{GDRIVE_PATH}sentiment_news")

Device set to use cuda:0


Processing 50000 news entries...


Industry Classification: 100%|██████████| 1563/1563 [55:03<00:00,  2.11s/batch]


Completed processing 50000 entries

Dropped 178 (Industry, Date) pairs with Industry='None'
Remaining pairs: 2207

Summary Statistics:
Total unique (Industry, Date) pairs: 2207
Average articles per pair: 21.65
Max articles in a pair: 803
Min articles in a pair: 1
25th percentile: 2.0
50th percentile: 9.0
75th percentile: 23.0
Number of pairs with at least 3 articles: 1622
Total articles: 47774


Extracting top 3 impactful news per (Industry, Date) pair...


Processing groups: 100%|██████████| 2207/2207 [00:00<00:00, 112846.87it/s]


Processing 2207 news entries...


FinBERT Sentiment: 100%|██████████| 9/9 [00:18<00:00,  2.02s/batch]


Completed processing 2207 entries


In [ ]:
final_df = await processor.get_explanation(df, save_path=f"{GDRIVE_PATH}explanation_news")

Explanation: 100%|██████████| 2207/2207 [11:07<00:00,  3.31it/s]


In [ ]:
df

,Industry,Date,News,ArticleCount,ImpactfulNews,AvgSentimentScore,SentimentScore,SentimentExplanation
0,Communication Services,2011-10-06,[{'Headline': 'FCC to Revamp Phone Subsidy to ...,2,[{'Headline': 'Euro-Area Leaders to Hold Summi...,0.709191,-0.291917,Overall sentiment for the Communications Servi...
1,Consumer Discretionary,2011-10-06,[{'Headline': 'PepsiCo May Purchase Russian Dr...,1,[{'Headline': 'PepsiCo May Purchase Russian Dr...,0.881740,0.888237,The article’s sentiment is strongly positive f...
2,Consumer Staples,2011-10-06,[{'Headline': 'Ukraine’s Grain Harvest Advance...,1,[{'Headline': 'Ukraine’s Grain Harvest Advance...,-0.918589,-0.917441,Explanation: FinBERT indicates a strongly nega...
3,Energy,2011-10-06,[{'Headline': 'Clean-Tech Companies Should Get...,9,[{'Headline': 'Norway Boosts Mongstad Carbon-S...,0.252093,-0.252850,"The energy-angle sentiment is mildly negative,..."
4,Financials,2011-10-06,[{'Headline': 'Ivory Coast Keeps Cocoa Export ...,51,[{'Headline': 'Remittances to Vietnam Thru Jul...,-0.306989,-0.032282,Combined FinBERT signal for the Financials set...
5,General Market,2011-10-06,[{'Headline': 'Farmland Seen Returning Up to 1...,4,[{'Headline': 'GE Study Finds Recession’s Job ...,-0.256293,-0.873807,Overall sentiment is negative: the combined Fi...
6,Health Care,2011-10-06,[{'Headline': 'House Panel Seeks Details on IR...,2,[{'Headline': 'Emdeon Said to Set Rate on $1.2...,0.666714,0.682417,Score: +0.67. Explanation: The Health Care new...
7,Industrials,2011-10-06,[{'Headline': 'Airbus German Workers Plan Work...,5,"[{'Headline': 'Polish Stocks: Getin, KGHM, Lot...",-0.296801,-0.868415,Explanation: The Industrials sentiment is nega...
8,Information Technology,2011-10-06,[{'Headline': 'Fans Hold IPhone-Lit Vigils for...,2,[{'Headline': 'Fans Hold IPhone-Lit Vigils for...,0.412332,0.002113,Net sentiment for the Information Technology t...
9,Materials,2011-10-06,[{'Headline': 'USDA Boxed Beef Cutout Closing ...,5,[{'Headline': 'Ukraine September Consumer Pric...,0.853697,0.875244,Combined materials-focused sentiment is strong...


In [ ]:
import pandas as pd
pd.read_parquet(f"{GDRIVE_PATH}explanation_news")

,Industry,Date,News,ArticleCount,ImpactfulNews,AvgSentimentScore,SentimentScore
0,Industrials,2006-10-20,"[{'Article': 'Inco Ltd., the Canadian nickel p...",1,"[{'Article': 'Inco Ltd., the Canadian nickel p...",-0.306854,-0.430676
1,Financials,2006-10-21,[{'Article': 'Jim Cramer recommended that view...,1,[{'Article': 'Jim Cramer recommended that view...,0.467539,0.463383
2,Financials,2007-01-03,"[{'Article': 'Kaye Scholer LLP, the 500-lawyer...",1,"[{'Article': 'Kaye Scholer LLP, the 500-lawyer...",0.878102,0.872732
3,Communication Services,2007-02-12,[{'Article': 'A federal judge declared Adelphi...,1,[{'Article': 'A federal judge declared Adelphi...,0.714189,0.751565
4,Financials,2007-02-12,[{'Article': 'Advanced Marketing Services Inc....,2,[{'Article': 'Advanced Marketing Services Inc....,0.185984,-0.779473
...,...,...,...,...,...,...,...
2202,Real Estate,2013-11-01,[{'Article': 'A three-way bidding war for Aust...,2,[{'Article': 'A three-way bidding war for Aust...,-0.180499,0.004677
2203,Utilities,2013-11-01,[{'Article': 'More than 30 months after an ear...,1,[{'Article': 'More than 30 months after an ear...,0.004326,0.001697
2204,Financials,2013-11-04,[{'Article': 'The U.S. jobs report in the comi...,1,[{'Article': 'The U.S. jobs report in the comi...,0.172938,0.230453
2205,Materials,2013-11-04,[{'Article': 'Grain exports from the French po...,1,[{'Article': 'Grain exports from the French po...,0.008498,0.009371


# Agent

In [7]:
!pip install feedparser

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 7.0 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=1bb1f66232196ee7a31d356c102ceba267f7596ffc06039e011f2b411d730809
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k


In [15]:
!pip install langchain langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 8.1 MB/s eta 0:00:00


In [67]:
from typing import List, Dict, Any
from datetime import datetime, timedelta, timezone
import feedparser
config = Config()

from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent


In [69]:
@tool
def get_bloomberg_rss_feeds(days: int = 1) -> List[Dict[str, str]]:
    """Fetch Bloomberg RSS news items for the last N days."""

    from config import Config
    config = Config()
    feeds = config.rss_feeds[:2]

    cutoff = datetime.now(timezone.utc) - timedelta(days=days)
    news = []

    for url in feeds:
        feed = feedparser.parse(url)
        for entry in feed.entries:
            if hasattr(entry, "published_parsed"):
                published = datetime(*entry.published_parsed[:6], tzinfo=timezone.utc)
                if published > cutoff:
                    news.append({
                        "Headline": entry.title,
                        # "Link": entry.link,
                        "Article": entry.summary,
                        # "Date": published.isoformat(),
                    })

    return news

In [70]:
get_bloomberg_rss_feeds(1)

[{'Headline': 'China Halts Some Brazil Soybean Imports Over Contamination',
  'Link': 'https://www.bloomberg.com/news/articles/2025-11-27/china-halts-some-brazil-soybean-imports-over-contamination',
  'Article': 'China halted soybean imports from five Brazilian plants owned by major global agricultural firms over sanitation concerns, according to people familiar with the matter.',
  'Date': '2025-11-27T14:44:51+00:00'},
 {'Headline': 'Mercuria, Vitol Are Among Bidders for Raizen Argentina Refinery',
  'Link': 'https://www.bloomberg.com/news/articles/2025-11-27/mercuria-vitol-are-among-bidders-for-raizen-argentina-refinery',
  'Article': 'Energy trading giants Mercuria Energy Group and Vitol Group are among finalists for a refinery and hundreds of gas stations in Argentina being sold by Raizen SA, according to people familiar with the matter.',
  'Date': '2025-11-27T19:41:50+00:00'},
 {'Headline': 'Global Stocks Hold Steady After Four-Day Rally: Markets Wrap',
  'Link': 'https://www.blo

In [31]:
@tool
def process_bloomberg_news(data: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Run the full NewsProcessor pipeline on raw Bloomberg RSS feed entries.
    Input: List of dicts with Headline, Link, Article, Date
    Output: Processed dataframe converted to list[dict]
    """
    from config import Config
    from processor import NewsProcessor

    config = Config()
    processor = NewsProcessor(config)

    # Pipeline
    data = processor.remove_redundant_info(data)
    df = processor.enrich_news_entries_with_classifications(data)
    df = processor.group_by_date_and_industry(df)
    df = processor.filter_and_analyze_news(df)
    df = processor.extract_impactful_news(df, top_n=3)
    df = processor.get_consolidated_sentiment(df)

    return df.to_dict(orient="records")

In [49]:
llm = ChatOpenAI(
    model="gpt-5-nano",
    temperature=0,
    streaming=True
)

tools = [
    get_bloomberg_rss_feeds,
    # process_bloomberg_news
]

# Create a ReAct agent using LangGraph
app = create_react_agent(
    model=llm,
    tools=tools,
)

/tmp/ipython-input-2917618498.py:13: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  app = create_react_agent(


In [57]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[get_bloomberg_rss_feeds],
)
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "Fetch Bloomberg RSS news from the past 1 day"}]},
    stream_mode="updates",
):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")

step: model
content: [{'type': 'tool_call', 'name': 'get_bloomberg_rss_feeds', 'args': {'days': 1}, 'id': 'call_VZIFnmNQbvwvgY9NDdeuBhTa'}]


/tmp/ipython-input-2928978895.py:10: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  cutoff = datetime.utcnow() - timedelta(days=days)


step: tools
content: [{'type': 'text', 'text': '[{"Headline": "China Halts Some Brazil Soybean Imports Over Contamination", "Link": "https://www.bloomberg.com/news/articles/2025-11-27/china-halts-some-brazil-soybean-imports-over-contamination", "Article": "China halted soybean imports from five Brazilian plants owned by major global agricultural firms over sanitation concerns, according to people familiar with the matter.", "Date": "2025-11-27T14:44:51"}, {"Headline": "Mercuria, Vitol Are Among Bidders for Raizen Argentina Refinery", "Link": "https://www.bloomberg.com/news/articles/2025-11-27/mercuria-vitol-are-among-bidders-for-raizen-argentina-refinery", "Article": "Energy trading giants Mercuria Energy Group and Vitol Group are among finalists for a refinery and hundreds of gas stations in Argentina being sold by Raizen SA, according to people familiar with the matter.", "Date": "2025-11-27T19:41:50"}, {"Headline": "Global Stocks Hold Steady After Four-Day Rally: Markets Wrap", "Lin

In [61]:
data["messages"][-1].content_blocks

[{'type': 'text',
  'text': "Here are Bloomberg RSS news items from the past day (timestamps shown as in the feed):\n\n1) China Halts Some Brazil Soybean Imports Over Contamination — 2025-11-27T14:44:51 — https://www.bloomberg.com/news/articles/2025-11-27/china-halts-some-brazil-soybean-imports-over-contamination\n2) Mercuria, Vitol Are Among Bidders for Raizen Argentina Refinery — 2025-11-27T19:41:50 — https://www.bloomberg.com/news/articles/2025-11-27/mercuria-vitol-are-among-bidders-for-raizen-argentina-refinery\n3) Global Stocks Hold Steady After Four-Day Rally: Markets Wrap — 2025-11-26T22:30:39 — https://www.bloomberg.com/news/articles/2025-11-26/stock-market-today-dow-s-p-live-updates\n4) Still Constructive on Chinese Equities, Huynh Says — 2025-11-27T19:06:21 — https://www.bloomberg.com/news/videos/2025-11-27/still-constructive-on-chinese-equities-huynh-says-video\n5) Oil Stalls as Traders Await Next Steps on Ukraine, OPEC+ Meeting — 2025-11-26T23:27:22 — https://www.bloomberg.